# Clase 4 — Entrenar el modelo

Curso **"Deep Learning con Python y PyTorch desde cero"** — DIH DATAlife, a partir de la certificación EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Duración estimada de esta clase:** 120 minutos.

## Objetivos de aprendizaje de esta clase

- Explicar el ciclo de entrenamiento: forward, cálculo de la pérdida (loss), backward y actualización de parámetros
- Elegir una función de pérdida y un optimizador adecuados para un problema de clasificación
- Implementar un bucle de entrenamiento completo con PyTorch
- Evaluar la evolución de la pérdida y la precisión (accuracy) durante el entrenamiento

Este notebook acompaña al capítulo 04 del manual del curso. Ejecuta cada celda en orden con **Mayús + Intro**. Retomamos exactamente la red neuronal (Neural Network, NN) que construiste en la clase 3: una entrada de 784 valores, capas `Linear` con activación `ReLU` y una salida de diez clases. Hoy, en cambio, esa red va a aprender de verdad.

## Parte 1 — El ciclo de entrenamiento

### Por qué una red recién creada no sirve para nada

Cuando creaste la red en la clase anterior, sus pesos eran números aleatorios. Si le pasas una imagen, obtienes una predicción, pero esa predicción no significa nada todavía: la red nunca vio un ejemplo. **Entrenar** es el proceso que convierte esos pesos aleatorios en pesos útiles.

### La analogía del grifo

Piensa en ajustar la temperatura de un grifo con dos mandos (frío y caliente) sin mirar ningún número: abres el grifo, tocas el agua, notas cuánto te alejas de la temperatura ideal y giras el mando en consecuencia. Repites el gesto, con ajustes cada vez más finos, hasta acercarte a lo que buscas.

Entrenar una red repite esa misma lógica miles de veces, con muchísimos más "mandos" (los pesos de la red). Cada vuelta del ciclo tiene cuatro pasos:

1. **Forward**: pasas un lote (batch) de datos por la red y obtienes una predicción
2. **Loss**: comparas esa predicción con la etiqueta real y obtienes un número que mide cuánto se equivocó la red
3. **Backward**: PyTorch calcula, para cada peso, en qué dirección y con qué intensidad debería cambiar para reducir ese error
4. **Optimizer.step**: aplicas esos cambios y actualizas los pesos

Repites el ciclo con el siguiente lote, y con el siguiente, hasta agotar el conjunto de datos: eso es una **época** (epoch). Entrenar durante varias épocas expone a la red a los mismos ejemplos más de una vez, para que refine sus pesos poco a poco en lugar de memorizar un solo repaso.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("Versión de PyTorch:", torch.__version__)

## Parte 2 — Recupera la red y los datos de las clases anteriores

### La red de la clase 3

Es la misma arquitectura: tres capas `Linear`, con `ReLU` entre medias, que recibe una imagen de MNIST aplanada en un vector de 784 valores (28x28 píxeles) y devuelve diez valores en bruto, uno por cada dígito posible.

In [ ]:
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        x = self.capa3(x)            # logits en bruto: la función de pérdida se encarga del resto
        return x

red = RedMNIST()
print(red)

### Los datos: MNIST con `torchvision` y `DataLoader`

Igual que en la clase 2, descargas MNIST con `torchvision.datasets.MNIST` y sirves los datos en lotes con `DataLoader`. Usamos el conjunto de entrenamiento para entrenar y el de prueba (test) para evaluar, más adelante, sobre imágenes que la red nunca vio.

In [ ]:
transformaciones = transforms.Compose([transforms.ToTensor()])

datos_entrenamiento = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
datos_prueba = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transformaciones
)

cargador_entrenamiento = torch.utils.data.DataLoader(datos_entrenamiento, batch_size=64, shuffle=True)
cargador_prueba = torch.utils.data.DataLoader(datos_prueba, batch_size=64, shuffle=False)

print("Imágenes de entrenamiento:", len(datos_entrenamiento))
print("Imágenes de prueba:", len(datos_prueba))

## Parte 3 — Función de pérdida y optimizador

- **Función de pérdida (loss)**: mide cuánto se aleja la predicción de la etiqueta real. Como este es un problema de clasificación con diez clases, usamos `nn.CrossEntropyLoss` (entropía cruzada), la elección habitual frente al error cuadrático medio (Mean Squared Error, MSE) que usarías en regresión
- **Optimizador**: ajusta los pesos de la red a partir de la pérdida y los gradientes. Usamos `Adam` (Adaptive Moment Estimation, estimación adaptativa de momentos), que adapta el tamaño de paso por cada peso automáticamente. La alternativa más simple es `SGD` (Stochastic Gradient Descent, descenso de gradiente estocástico), con un tamaño de paso fijo

El tamaño de paso se llama **tasa de aprendizaje** (learning rate). Con Adam, un valor de partida razonable es 0,001 (también escrito como 1e-3). Una tasa demasiado alta hace que el entrenamiento oscile sin converger; una demasiado baja lo hace innecesariamente lento.

In [ ]:
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(red.parameters(), lr=0.001)

print(funcion_perdida)
print(optimizador)

## Parte 4 — El bucle de entrenamiento completo

Fíjate en que cada uno de los cuatro pasos del ciclo que viste en la parte 1 aparece como una única línea:

- `optimizador.zero_grad()`: reinicia los gradientes acumulados del paso anterior. PyTorch acumula gradientes por defecto, así que si olvidas este paso, el entrenamiento deja de tener sentido
- `red(imagenes)`: forward, la red genera una predicción
- `funcion_perdida(salida, etiquetas)`: loss, cuánto se equivocó la red
- `perdida.backward()`: backward, calcula el gradiente de cada peso
- `optimizador.step()`: optimizer.step, actualiza los pesos

Entrenamos durante cinco épocas e imprimimos la pérdida media de cada una para comprobar que baja.

In [ ]:
epocas = 5
historial_perdida = []

for epoca in range(epocas):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador.zero_grad()
        salida = red(imagenes)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_epoca += perdida.item()

    perdida_media = perdida_epoca / len(cargador_entrenamiento)
    historial_perdida.append(perdida_media)
    print(f"Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}")

Visualiza cómo baja la pérdida época a época. Una curva descendente es la señal de que el modelo está aprendiendo.

In [ ]:
plt.plot(range(1, epocas + 1), historial_perdida, marker="o")
plt.xlabel("Época")
plt.ylabel("Pérdida media")
plt.title("Evolución de la pérdida durante el entrenamiento")
plt.show()

## Parte 5 — Evaluar la precisión sobre datos que la red no vio

Una pérdida que baja es una buena señal, pero lo que de verdad importa es si la red clasifica bien imágenes nuevas. Usamos el conjunto de prueba y el gestor de contexto `torch.no_grad()`, que evita calcular gradientes porque en esta fase solo medimos, no entrenamos.

`torch.argmax` se queda con el índice del valor más alto de los diez que devuelve la red para cada imagen: el dígito que considera más probable. Comparar esa predicción con la etiqueta real, sumar los aciertos y dividir entre el total te da la precisión (accuracy).

In [ ]:
correctos = 0
total = 0

with torch.no_grad():
    for imagenes, etiquetas in cargador_prueba:
        salida = red(imagenes)
        predicciones = torch.argmax(salida, dim=1)
        correctos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

precision = correctos / total
print(f"Precisión sobre el conjunto de prueba: {precision:.1%}")

### Un vistazo a las predicciones

Toma un lote del conjunto de prueba y compara, imagen a imagen, la etiqueta real con la predicción de la red.

In [ ]:
imagenes, etiquetas = next(iter(cargador_prueba))
with torch.no_grad():
    predicciones = torch.argmax(red(imagenes), dim=1)

figura, ejes = plt.subplots(1, 6, figsize=(12, 3))
for i, eje in enumerate(ejes):
    eje.imshow(imagenes[i][0], cmap="gray")
    eje.set_title(f"real: {etiquetas[i].item()} / pred: {predicciones[i].item()}")
    eje.axis("off")
plt.show()

## Resumen de la clase

- Una red recién creada tiene pesos aleatorios; entrenar es el proceso que los ajusta a partir de datos
- El ciclo de entrenamiento repite cuatro pasos por cada lote: forward, loss, backward y optimizer.step
- `nn.CrossEntropyLoss` mide el error en problemas de clasificación con varias clases; `Adam` y `SGD` son los optimizadores más habituales para corregirlo
- `optimizador.zero_grad()` antes de cada `backward()` es imprescindible, porque PyTorch acumula gradientes por defecto
- Una época es una pasada completa por el conjunto de entrenamiento; se necesitan varias para que la pérdida baje de forma consistente
- La precisión (accuracy) se mide sobre el conjunto de prueba, dentro de un bloque `torch.no_grad()`, para comprobar que la red generaliza y no solo memoriza

En el notebook de práctica vas a completar tú mismo un bucle de entrenamiento y a comparar optimizadores.